In [0]:
display(dbutils.fs.ls("/Volumes/airbnb_joao_pessoa/bronze/raw_files"))

In [0]:
# Caminho base
base_path = "/Volumes/airbnb_joao_pessoa/bronze/raw_files"

# Ler os quatro datasets
listings = spark.read.parquet(
    f"{base_path}/Listings_Data.parquet"
)

past_rates = spark.read.parquet(
    f"{base_path}/Past_Calendar_Rates.parquet"
)

future_rates = spark.read.parquet(
    f"{base_path}/Future_Calendar_Rates.parquet"
)

reviews = spark.read.parquet(
    f"{base_path}/Reviews_Data.parquet"
)

# Quantidade de registros
print("Listings:", listings.count())
print("Past Calendar Rates:", past_rates.count())
print("Future Calendar Rates:", future_rates.count())
print("Reviews:", reviews.count())

In [0]:
print("LISTINGS")
listings.printSchema()

print("\nPAST CALENDAR RATES")
past_rates.printSchema()

print("\nFUTURE CALENDAR RATES")
future_rates.printSchema()

print("\nREVIEWS")
reviews.printSchema()

In [0]:
print("Colunas Listings:", len(listings.columns))
print("Colunas Past Rates:", len(past_rates.columns))
print("Colunas Future Rates:", len(future_rates.columns))
print("Colunas Reviews:", len(reviews.columns))

Investigando a granularidade dos dados

In [0]:
# 1. Quantos listings aparecem nos Past Rates?
print(
    "Listings únicos no Past Rates:",
    past_rates.select("listing_id").distinct().count()
)

In [0]:
# 2. Quantos aparecem nos Future Rates?
print(
    "Listings únicos no Future Rates:",
    future_rates.select("listing_id").distinct().count()
)

In [0]:
# 3. Quantos listings têm reviews?
print(
    "Listings únicos nos Reviews:",
    reviews.select("listing_id").distinct().count()
)

In [0]:
# Período dos RATES
past_rates.selectExpr(
    "min(date) as primeira_data",
    "max(date) as ultima_data"
).show()

In [0]:
future_rates.selectExpr(
    "min(date) as primeira_data",
    "max(date) as ultima_data"
).show()

In [0]:
# Quantas observações existem por listing
from pyspark.sql.functions import count

display(
    past_rates
    .groupBy("listing_id")
    .agg(count("*").alias("qtd_registros"))
    .orderBy("qtd_registros", ascending=False)
)

In [0]:
# Distrbuição da ocupação
display(
    past_rates.select(
        "listing_id",
        "date",
        "occupancy",
        "revenue",
        "rate_avg"
    ).orderBy("date")
)

In [0]:
from pyspark.sql.functions import count, min, max

past_summary = (
    past_rates
    .groupBy("listing_id")
    .agg(
        count("*").alias("qtd_meses"),
        min("date").alias("primeiro_mes"),
        max("date").alias("ultimo_mes")
    )
    .orderBy("listing_id")
)

display(past_summary)

In [0]:
display(
    past_summary
    .groupBy("qtd_meses")
    .count()
    .orderBy("qtd_meses")
)

In [0]:
future_summary = (
    future_rates
    .groupBy("listing_id")
    .agg(
        count("*").alias("qtd_meses"),
        min("date").alias("primeiro_mes"),
        max("date").alias("ultimo_mes")
    )
    .orderBy("listing_id")
)

display(future_summary)

In [0]:
display(
    future_summary
    .groupBy("qtd_meses")
    .count()
    .orderBy("qtd_meses")
)